# Baseline Cox PH: clinical / expression / combined

In [8]:
import pandas as pd, numpy as np
from lifelines import CoxPHFitter
from lifelines.utils import k_fold_cross_validation
from lifelines.statistics import logrank_test

clin = pd.read_csv('data/processed/clinical.csv', index_col=0)
expr = pd.read_csv('data/processed/expr.csv', index_col=0)
comb = pd.read_csv('data/processed/combined.csv', index_col=0)
print(clin.shape, expr.shape, comb.shape); clin.head()

(183, 18) (176, 500) (176, 518)


,duration,event,age,male,stage,t_stage,n_positive,m_positive,neoadjuvant,radiation_therapy,prior_dx,weight,site_C25.0,site_C25.1,site_C25.2,site_C25.9,hist_8140/3,hist_8500/3
PATIENT_ID,,,,,,,,,,,,,,,,,,
TCGA-2J-AAB1,2.169839,1,65,1,NaN,3.0,1,0,0,0,0,NaN,1,0,0,0,0,0
TCGA-2J-AAB4,23.966861,0,48,1,NaN,2.0,1,0,0,0,0,NaN,0,0,0,0,1,0
TCGA-2J-AAB6,9.632771,1,75,1,NaN,3.0,0,0,0,0,0,NaN,0,1,0,0,0,1
TCGA-2J-AAB8,2.630108,0,71,1,NaN,3.0,1,0,0,0,0,NaN,1,0,0,0,0,1
TCGA-2J-AAB9,20.613473,1,70,0,NaN,3.0,1,0,0,1,0,NaN,1,0,0,0,0,1


In [9]:
from sklearn.model_selection import KFold
from lifelines.utils import concordance_index

def prep(df, min_count=10):
    y = df[['duration','event']]; X = df.drop(columns=['duration','event'])
    X = X.dropna(axis=1, how='all')           # drop columns that are entirely empty
    X = X.fillna(X.median())                  # fill remaining gaps with the median
    binary = [c for c in X if X[c].nunique() == 2]
    X = X.drop(columns=[c for c in binary if X[c].sum() < min_count or (1 - X[c]).sum() < min_count])
    X = X.loc[:, X.std() > 0]                 # drop constant columns
    return pd.concat([y, X], axis=1)

ids = comb.index
sets = {'clinical': prep(clin.loc[ids]),
        'expression': prep(comb[['duration','event'] + list(expr.columns)]),
        'combined': prep(comb)}
PEN = 0.1

def cv_cindex(df, k=10, seed=42):
    scores = []
    for tr, te in KFold(k, shuffle=True, random_state=seed).split(df):
        d_tr, d_te = df.iloc[tr].copy(), df.iloc[te].copy()
        feats = [c for c in df if c not in ('duration','event')]
        mu, sd = d_tr[feats].mean(), d_tr[feats].std().replace(0, 1)   # scale on train only
        d_tr[feats] = (d_tr[feats] - mu) / sd
        d_te[feats] = (d_te[feats] - mu) / sd
        m = CoxPHFitter(penalizer=PEN, l1_ratio=0.0).fit(d_tr, 'duration', 'event')
        risk = m.predict_partial_hazard(d_te)
        scores.append(concordance_index(d_te['duration'], -risk, d_te['event']))
    return scores

rows = []
for name, df in sets.items():
    s = cv_cindex(df)
    rows.append((name, df.shape[1]-2, np.mean(s), np.std(s)))
res = pd.DataFrame(rows, columns=['features','n_features','cv_c_index','std'])
res.to_csv('data/processed/cindex_table.csv', index=False); res

,features,n_features,cv_c_index,std
0,clinical,12,0.625715,0.120408
1,expression,500,0.569889,0.083088
2,combined,512,0.579458,0.074758


In [10]:
# hand-off for teammate: fit on all data, save risk scores + model
import pickle
N, PEN_FINAL = 100, 1.0            # change if 150/200 clearly won
g = list(expr.columns[:N])
df = prep(comb[['duration', 'event'] + clin_cols + g])
feats = [c for c in df if c not in ('duration', 'event')]
sc = df.copy()
sc[feats] = (sc[feats] - sc[feats].mean()) / sc[feats].std().replace(0, 1)

cph = CoxPHFitter(penalizer=PEN_FINAL, l1_ratio=0.0).fit(sc, 'duration', 'event')
risk = cph.predict_partial_hazard(sc)
out = sc[['duration', 'event']].assign(risk_score=risk)
out.to_csv('data/processed/risk_scores.csv')
pickle.dump(cph, open('data/processed/cox_combined.pkl', 'wb'))

hi = out.risk_score > out.risk_score.median()
print(logrank_test(out.duration[hi], out.duration[~hi], out.event[hi], out.event[~hi]).p_value)

6.607198729639546e-12


In [7]:
from sklearn.model_selection import KFold

def cv_cindex(df, pen, k=10, seed=42):
    scores = []
    feats = [c for c in df if c not in ('duration', 'event')]
    for tr, te in KFold(k, shuffle=True, random_state=seed).split(df):
        d_tr, d_te = df.iloc[tr].copy(), df.iloc[te].copy()
        mu, sd = d_tr[feats].mean(), d_tr[feats].std().replace(0, 1)
        d_tr[feats] = (d_tr[feats] - mu) / sd
        d_te[feats] = (d_te[feats] - mu) / sd
        m = CoxPHFitter(penalizer=pen, l1_ratio=0.0).fit(d_tr, 'duration', 'event')
        scores.append(concordance_index(d_te['duration'], -m.predict_partial_hazard(d_te), d_te['event']))
    return np.mean(scores)

PENS, GENES, SEEDS = [0.1, 1, 10, 50], [25, 50, 100], [1, 2, 3]
clin_cols = [c for c in clin.columns if c not in ('duration', 'event')]
rows = []

# clinical: sweep penalty only
for pen in PENS:
    df = prep(clin.loc[comb.index])
    rows.append(('clinical', 0, pen, np.mean([cv_cindex(df, pen, seed=s) for s in SEEDS])))

# expression + combined: sweep penalty x gene count
for n in GENES:
    g = list(expr.columns[:n])
    for name, cols in [('expression', g), ('combined', clin_cols + g)]:
        df = prep(comb[['duration', 'event'] + cols])
        for pen in PENS:
            rows.append((name, n, pen, np.mean([cv_cindex(df, pen, seed=s) for s in SEEDS])))
            print(name, n, pen, round(rows[-1][3], 3))

sweep = pd.DataFrame(rows, columns=['features', 'n_genes', 'penalizer', 'cv_c_index'])
sweep.to_csv('data/processed/sweep_results.csv', index=False)
sweep.sort_values('cv_c_index', ascending=False).groupby('features').head(3)

expression 25 0.1 0.497
expression 25 1 0.488
expression 25 10 0.506
expression 25 50 0.51
combined 25 0.1 0.602
combined 25 1 0.601
combined 25 10 0.571
combined 25 50 0.559
expression 50 0.1 0.585
expression 50 1 0.524
expression 50 10 0.513
expression 50 50 0.509
combined 50 0.1 0.634
combined 50 1 0.596
combined 50 10 0.557
combined 50 50 0.539
expression 100 0.1 0.629
expression 100 1 0.638
expression 100 10 0.602
expression 100 50 0.579
combined 100 0.1 0.664
combined 100 1 0.672
combined 100 10 0.625
combined 100 50 0.598


,features,n_genes,penalizer,cv_c_index
25,combined,100,1.0,0.671818
24,combined,100,0.1,0.664293
21,expression,100,1.0,0.638202
16,combined,50,0.1,0.634498
20,expression,100,0.1,0.628984
0,clinical,0,0.1,0.626157
1,clinical,0,1.0,0.623193
3,clinical,0,50.0,0.616732
22,expression,100,10.0,0.602361


In [11]:
r = pd.read_csv('data/processed/risk_scores.csv', index_col=0)
print(r.shape, len(cph.params_))

(176, 3) 112
